In [1]:
pip install pandas pyarrow

Note: you may need to restart the kernel to use updated packages.


In [5]:
import pandas as pd

jan = pd.read_parquet("yellow_tripdata_2026-01.parquet")

feb = pd.read_parquet("yellow_tripdata_2026-02.parquet")

mar = pd.read_parquet("yellow_tripdata_2026-03.parquet")

combined = pd.concat([jan, feb, mar], ignore_index=True)

print(f"Total rows: {len(combined):,}")
print(f"Total columns: {combined.shape[1]}")

print("Saving to CSV...")
combined.to_csv("combined.csv", index=False)

Total rows: 11,077,206
Total columns: 20
Saving to CSV...


In [7]:
df=pd.read_csv('combined.csv')

/var/folders/d9/msr7g525037gq7yzyxm24kkc0000gn/T/ipykernel_19196/4059180406.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('combined.csv')


In [8]:
#CLEANING THE DATASET

#1) CONVERTING THE TIMESTAMPS
df['tpep_pickup_datetime'] = pd.to_datetime(df['tpep_pickup_datetime'])
df['tpep_dropoff_datetime'] = pd.to_datetime(df['tpep_dropoff_datetime'])

#2) CONVERTING TRIP DURATION TO MINUTES
df['trip_duration'] = (
    df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
).dt.total_seconds() / 60

#3) KEEPING THE FARE AMOUNT BETWEEN $2.5 AND $500
df = df[(df['fare_amount'] >= 2.5) & (df['fare_amount'] <= 500)]

#4) KEEPING THE DISTANCE BETWEEN 0.1 AND 100
df = df[(df['trip_distance'] >= 0.1) & (df['trip_distance'] <= 100)]

#5) KEEPING THE TIME DURATION BETWEEN 1 MIN AND 300 MIN
df = df[(df['trip_duration'] >= 1) & (df['trip_duration'] <= 300)]

#6) REMOVING THE INVALID LOCATIONS
df = df[
    df['PULocationID'].notna() &
    df['DOLocationID'].notna() &
    ~df['PULocationID'].isin([264, 265]) &
    ~df['DOLocationID'].isin([264, 265])
]

#7) KEEPING PASSENGER COUNT MORE THAN 0
df = df[df['passenger_count'] > 0]

# Save cleaned dataset
df.to_csv("combined_cleaned.csv", index=False)

print(df.shape)

(7576121, 21)


In [9]:
import pandas as pd
import numpy as np

In [11]:
#SLICING HOUR OF THE DAY
df['hour_of_day'] = df['tpep_pickup_datetime'].dt.hour

#SLICING DAY OF THE WEEK
df['day_of_week'] = df['tpep_pickup_datetime'].dt.dayofweek

#SLICING WEEKEND FLAG
df['is_weekend'] = df['day_of_week'].isin([5, 6])

#SLICING PEAK HOUR FLAG
df['is_peak'] = df['hour_of_day'].isin(
    [7, 8, 9, 16, 17, 18, 19]
)

#CALCULATING REVENUE PER MILE
df['revenue_per_mile'] = (
    df['total_amount'] / df['trip_distance']
)

# Speed (mph)
df['speed_mph'] = (
    df['trip_distance'] /
    (df['trip_duration'] / 60)
)

#CALCULATING TIP PERCENTAGE
df['tip_pct'] = np.where(
    df['fare_amount'] > 0,
    df['tip_amount'] / df['fare_amount'] * 100,
    0
)

#CALCULATING AIRPORT TRIPS
df['is_airport'] = df['RatecodeID'].isin([2, 3])

#TIME INTELLIGENCE
df['month'] = df['tpep_pickup_datetime'].dt.month
df['quarter'] = df['tpep_pickup_datetime'].dt.quarter
df['year'] = df['tpep_pickup_datetime'].dt.year

In [12]:
df.to_csv("combined_cleaned.csv", index=False)

In [13]:
df

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,day_of_week,is_weekend,is_peak,revenue_per_mile,speed_mph,tip_pct,is_airport,month,quarter,year
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,...,3,False,False,16.350515,10.486486,50.833333,False,1,1,2026
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,...,3,False,False,9.956989,7.822430,28.708010,False,1,1,2026
5,2,2026-01-01 00:47:11,2026-01-01 01:00:47,2.0,2.33,1.0,N,144,137,1,...,3,False,False,10.703863,10.279412,35.140845,False,1,1,2026
6,1,2026-01-01 00:17:54,2026-01-01 00:28:32,1.0,1.30,1.0,N,142,50,2,...,3,False,False,13.192308,7.335423,0.000000,False,1,1,2026
8,2,2026-01-01 00:34:14,2026-01-01 01:11:58,1.0,5.34,1.0,N,161,45,1,...,3,False,False,9.674157,8.491166,23.083110,False,1,1,2026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10131453,2,2026-03-31 23:17:20,2026-03-31 23:27:29,1.0,2.29,1.0,N,236,163,1,...,1,False,False,10.126638,13.536946,36.250000,False,3,1,2026
10131454,2,2026-03-31 23:30:28,2026-03-31 23:40:04,1.0,1.90,1.0,N,163,246,1,...,1,False,False,10.831579,11.875000,30.087719,False,3,1,2026
10131455,2,2026-03-31 23:15:21,2026-03-31 23:30:37,1.0,2.62,1.0,N,142,107,1,...,1,False,False,9.263359,10.296943,13.619632,False,3,1,2026
10131456,2,2026-03-31 23:02:01,2026-03-31 23:06:57,1.0,1.03,1.0,N,170,100,1,...,1,False,False,12.572816,12.527027,0.000000,False,3,1,2026
